# TABLA DE INVENTARIO (OINM) SEMANAL 

**Autor: Abdias Figueredo V.**

**Version: 1.0**

**Agosto 2025**

**Cochabamba, Bolivia**

## Índice
1.  **Introducción**

2.  **Configuración Inicial**

3.  **Analisis Univariado**

4.  **Análisis de Distribuciones**

5.  **Análisis de Outliers**

7.  **Conclusiones**

## 1. INTRODUCCIÓN

Este proyecto tiene como objetivo realizar un Análisis Exploratorio de Datos (EDA) exhaustivo de los datos de ventas, productos e inventario de una empresa importadora de autopartes. La empresa maneja alrededor de 10,000 SKUs diferentes y tiene registros desde 2018 hasta la fecha actual, generados con IA simulando un sistema SAP Business One.

**Objetivos del EDA:**

*   Comprender la estructura y calidad de los datos disponibles.
*   Identificar patrones y tendencias en las ventas a lo largo del tiempo.
*   Analizar el comportamiento de los diferentes SKUs en términos de ventas y rotación de inventario.
*   Detectar posibles problemas en los datos, como valores atípicos o inconsistencias.
*   Generar hipótesis para futuros análisis y modelado predictivo.

**Tablas de Datos:**

*   **OITM (Tabla de Productos):** Contiene información detallada sobre cada producto, incluyendo su código, descripción, precio, etc.
*   **INV1 (Tabla de Ventas):** Registra las transacciones de venta, incluyendo la fecha, el SKU vendido, la cantidad, el precio, etc.
*   **OINM (Tabla de Inventario):** Contiene información sobre los movimientos de inventario, como entradas, salidas y ajustes.

---

## 2. CONFIGURACION INICIAL

### 2.1 Importación de librerías necesarias

In [1]:
# INSTALACION DE REQUIREMENTS
# Comando alternativo: pip install -r requirements.txt
#!pip install -r requirements.txt --quiet
# pip install --only-binary :all: statsforecast

In [2]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# =============================================================================
# Manejo y procesamiento de datos
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
from pandas.tseries.offsets import DateOffset

# Análisis de series temporales
# -----------------------------------------------------------------------------
from statsforecast import StatsForecast
from utilsforecast.plotting import plot_series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Visualización
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Configuración de advertencias
# -----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilos de visualización
# =============================================================================
# Configuración general de matplotlib
plt.style.use('classic')
plt.rcParams.update({
    'figure.figsize': (18, 7),
    'axes.facecolor': '#FFFFFF',  # Fondo blanco para mejor legibilidad
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16
})

# Configuración de seaborn
#sns.set_theme(style="whitegrid", 
#              rc={'axes.facecolor': '#FFFFFF'},
#              font_scale=1.1)

c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.2 Carga de datos

In [3]:
# =============================================================================
# CARGA DE DATOS
# =============================================================================
# Definición de rutas de archivos
DATA_DIR = '../data/bronze'
FILE_PATHS = {
    'oitm': os.path.join(DATA_DIR, 'oitm.parquet'),
    'inv1': os.path.join(DATA_DIR, 'inv1.parquet'),
    'oinm': os.path.join(DATA_DIR, 'oinm.parquet')
}

# Carga de datos con manejo de errores
try:
    # Cargar datos en un diccionario para mejor organización
    data = {
        'oitm': pd.read_parquet(FILE_PATHS['oitm']),
        'inv1': pd.read_parquet(FILE_PATHS['inv1']),
        'oinm': pd.read_parquet(FILE_PATHS['oinm'])
    }
    
    # Crear copias para procesamiento
    df_oitm = data['oitm'].copy()
    df_inv1 = data['inv1'].copy()
    df_oinm = data['oinm'].copy()
    
    # Verificación de carga exitosa
    print("✅ Datos cargados exitosamente")
    print(f"📊 Registros cargados:")
    print(f"   - oitm: {len(df_oitm):,} registros")
    print(f"   - inv1: {len(df_inv1):,} registros")
    print(f"   - oinm: {len(df_oinm):,} registros")
    
except Exception as e:
    print(f"❌ Error al cargar los datos: {str(e)}")
    # Opcional: cargar datos de respaldo o finalizar la ejecución
    raise

✅ Datos cargados exitosamente
📊 Registros cargados:
   - oitm: 10,000 registros
   - inv1: 1,000,000 registros
   - oinm: 1,055,133 registros


---

### 3.3 Descripcion de la Tabla de inventario (OINM)

In [4]:
# FUNCION PARA MOSTRAR INFORMACIÓN DEL DATAFRAME

def display_dataframe_info(df, name, rows=5):
    """
    Muestra información detallada de un DataFrame de manera formateada.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame a mostrar
    name : str
        Nombre descriptivo del DataFrame
    rows : int, opcional
        Número de filas a mostrar (por defecto: 5)
    """
    # Título con formato
    print(f"\n{'='*80}")
    print(f"📋 INFORMACIÓN DEL DATAFRAME: {name.upper()}")
    print(f"{'='*80}")
    
    # Mostrar dimensiones
    print(f"📊 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    
    # Mostrar tipos de datos
    print("\n📝 Tipos de datos:")
    print(df.dtypes)
    
    # Mostrar primeras filas
    print(f"\n🔍 Primeras {rows} filas:")
    display(df.head(rows))
    
    # Mostrar estadísticas descriptivas
    print("\n📈 Estadísticas descriptivas:")
    display(df.describe(include='all'))
    
    # Mostrar valores nulos
    print("\n❓ Valores nulos por columna:")
    null_counts = df.isnull().sum()
    null_percent = (df.isnull().mean() * 100).round(2)
    null_df = pd.DataFrame({
        'Valores nulos': null_counts,
        'Porcentaje (%)': null_percent
    })
    display(null_df[null_df['Valores nulos'] > 0] if null_df['Valores nulos'].sum() > 0 
            else "✅ No hay valores nulos")

In [5]:
# Uso de la función
display_dataframe_info(df_oinm, "Inventario (OINM)")


📋 INFORMACIÓN DEL DATAFRAME: INVENTARIO (OINM)
📊 Dimensiones: 1,055,133 filas x 8 columnas

📝 Tipos de datos:
TransNum               int64
TransType             object
DocDate       datetime64[ns]
ItemCode              object
OutQty                 int64
InQty                  int64
Price                float64
IsStockout             int64
dtype: object

🔍 Primeras 5 filas:


,TransNum,TransType,DocDate,ItemCode,OutQty,InQty,Price,IsStockout
0,1,IN,2017-12-25,SKU-00000,0,56,131.44,0
1,2,IN,2017-12-25,SKU-00001,0,110,242.43,0
2,3,OUT,2018-02-15,SKU-00001,8,0,116.02,0
3,4,OUT,2018-04-11,SKU-00001,7,0,57.77,0
4,5,OUT,2018-06-26,SKU-00001,8,0,321.58,0



📈 Estadísticas descriptivas:


,TransNum,TransType,DocDate,ItemCode,OutQty,InQty,Price,IsStockout
count,1.055133e+06,1055133,1055133,1055133,1.055133e+06,1.055133e+06,1.055133e+06,1.055133e+06
unique,NaN,2,NaN,10000,NaN,NaN,NaN,NaN
top,NaN,OUT,NaN,SKU-00306,NaN,NaN,NaN,NaN
freq,NaN,1000000,NaN,444,NaN,NaN,NaN,NaN
mean,5.275670e+05,NaN,2021-10-09 12:34:36.327439616,NaN,5.678001e+00,6.493334e+00,2.524539e+02,8.729326e-02
min,1.000000e+00,NaN,2017-12-25 00:00:00,NaN,0.000000e+00,0.000000e+00,4.050000e+00,0.000000e+00
25%,2.637840e+05,NaN,2019-11-11 00:00:00,NaN,2.000000e+00,0.000000e+00,1.307400e+02,0.000000e+00
50%,5.275670e+05,NaN,2021-10-08 00:00:00,NaN,5.000000e+00,0.000000e+00,2.520500e+02,0.000000e+00
75%,7.913500e+05,NaN,2023-09-08 00:00:00,NaN,8.000000e+00,0.000000e+00,3.731200e+02,0.000000e+00
max,1.055133e+06,NaN,2025-08-08 00:00:00,NaN,9.900000e+01,1.990000e+02,5.000000e+02,1.000000e+00



❓ Valores nulos por columna:


'✅ No hay valores nulos'

### 3.3 Limpieza inicial

In [6]:
# INNER JOIN ENTRE OINM Y OITM PARA FILTRAR ITEMS INACTIVOS Y GRUPOS
df_oinm = df_oinm.merge(df_oitm[['ItemCode', 'ItemGrp', 'FrozenFor']], 
                        on='ItemCode', 
                        how='inner')

In [7]:
# FILTRAR PARA QUITAR ITEMS INACTIVOS Y GRUPOS
def filter_active_items(df):
    """
    Filtra el DataFrame para eliminar items inactivos y grupos.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con columnas 'FrozenFor' y 'ItemGrp'
        
    Retorna:
    --------
    pandas.DataFrame
        DataFrame filtrado sin items inactivos ni grupos
    """
    # Filtrar items activos (no congelados) y grupos (no vacíos)
    filtered_df = df[(df['FrozenFor'] == 'N') & (df['ItemGrp'] != 124)]
    
    # Verificar si se eliminaron filas
    if len(filtered_df) < len(df):
        print(f"⚠️ Se eliminaron {len(df) - len(filtered_df):,} filas de items inactivos o grupos vacíos")
    
    return filtered_df

In [8]:
# USO DE LA FUNCION
df_oinm = filter_active_items(df_oinm)

⚠️ Se eliminaron 80,337 filas de items inactivos o grupos vacíos


In [9]:
# MOSTRAR LOS GRUPOS UNICOS
print("📋 ItemGrp únicos:"
      , df_oinm['ItemGrp'].unique())

📋 ItemGrp únicos: [809 202 405 303 101 607 708 506]


### Calculo de inventarios disponible semanal

In [10]:
# # FUNCION PARA PROCESAR INVENTARIO SEMANAL

# def process_weekly_inventory(df):
#     """
#     Procesa movimientos diarios de inventario usando isocalendar para semanas ISO 8601.
    
#     Parámetros:
#     -----------
#     df : pandas.DataFrame
#         DataFrame con las columnas:
#         - ItemCode: código del producto
#         - DocDate: fecha del movimiento
#         - InQty: cantidad de entrada
#         - OutQty: cantidad de salida
    
#     Retorna:
#     --------
#     pandas.DataFrame
#         Resumen semanal con formato ISO 8601
#     """
#     # 1. Preparación de fechas con isocalendar
#     df['DocDate'] = pd.to_datetime(df['DocDate'])
#     df['IsoYear'] = df['DocDate'].dt.isocalendar().year
#     df['IsoWeek'] = df['DocDate'].dt.isocalendar().week
#     df['Week'] = df['IsoYear'].astype(str) + '-W' + df['IsoWeek'].astype(str).str.zfill(2)
#     df['StartWeek'] = df['DocDate'] - pd.to_timedelta(df['DocDate'].dt.isocalendar().day - 1, unit='D')
    
#     # 2. Agrupación semanal
#     weekly = (df.groupby(['ItemCode', 'Week', 'StartWeek', 'IsoYear', 'IsoWeek'])
#              .agg({
#                  'InQty': 'sum',
#                  'OutQty': 'sum'
#              })
#              .reset_index())
    
#     # 3. Cálculo de movimiento neto
#     weekly['NetMovement'] = weekly['InQty'] - weekly['OutQty']
    
#     # 4. Crear todas las combinaciones de semanas
#     start_date = weekly['StartWeek'].min()
#     end_date = weekly['StartWeek'].max()
    
#     # Usar freq='W-MON' para asegurar que las semanas empiecen en lunes
#     all_weeks = pd.date_range(start=start_date, 
#                             end=end_date, 
#                             freq='W-MON')
    
#     items = weekly['ItemCode'].unique()
    
#     # 5. Producto cartesiano de items y semanas
#     complete_df = pd.DataFrame(
#         [(item, week) for item in items for week in all_weeks],
#         columns=['ItemCode', 'StartWeek']
#     )
    
#     # Añadir información ISO a complete_df
#     complete_df['IsoYear'] = complete_df['StartWeek'].dt.isocalendar().year
#     complete_df['IsoWeek'] = complete_df['StartWeek'].dt.isocalendar().week
#     complete_df['Week'] = (complete_df['IsoYear'].astype(str) + '-W' + 
#                           complete_df['IsoWeek'].astype(str).str.zfill(2))
    
#     # 6. Unión con datos existentes
#     weekly = pd.merge(
#         complete_df,
#         weekly[['ItemCode', 'Week', 'InQty', 'OutQty', 'NetMovement']],
#         on=['ItemCode', 'Week'],
#         how='left'
#     )
    
#     # 7. Rellenar valores faltantes
#     weekly[['InQty', 'OutQty', 'NetMovement']] = weekly[['InQty', 'OutQty', 'NetMovement']].fillna(0)
    
#     # 8. Cálculo de stock disponible
#     weekly['AvailableStock'] = weekly.groupby('ItemCode')['NetMovement'].cumsum()
    
#     # 9. Detección de stockouts
#     weekly['Stockout'] = (weekly['AvailableStock'] <= 0).astype(int)
    
#     # 10. Ordenar y seleccionar columnas finales
#     result = weekly[['ItemCode', 'StartWeek', 'Week', 'InQty', 'OutQty', 
#                     'NetMovement', 'AvailableStock', 'Stockout']]
    
#     return result.sort_values(['ItemCode', 'StartWeek'])

In [11]:
# # Ejemplo de uso
# dfWeekStock = process_weekly_inventory(df_oinm)
# print("Muestra del inventario procesado:")
# display(dfWeekStock.head(10))

In [12]:
# FUNCION PARA PROCESAR INVENTARIO SEMANAL

def process_weekly_inventory(df):
    """
    Procesa movimientos diarios de inventario usando isocalendar para semanas ISO 8601,
    considerando solo semanas completas.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con las columnas ItemCode, DocDate, InQty, OutQty
    
    Retorna:
    --------
    pandas.DataFrame
        Resumen semanal solo con semanas completas
    """
    # 1. Preparación de fechas con isocalendar
    df['DocDate'] = pd.to_datetime(df['DocDate'])
    df['IsoYear'] = df['DocDate'].dt.isocalendar().year
    df['IsoWeek'] = df['DocDate'].dt.isocalendar().week
    df['Week'] = df['IsoYear'].astype(str) + '-W' + df['IsoWeek'].astype(str).str.zfill(2)
    df['StartWeek'] = df['DocDate'] - pd.to_timedelta(df['DocDate'].dt.isocalendar().day - 1, unit='D')
    
    # 2. Identificar la última semana completa
    max_date = df['DocDate'].max()
    days_to_end_of_week = 7 - max_date.isocalendar().weekday
    
    if days_to_end_of_week < 7:
        # Usar isocalendar para la última semana completa
        last_complete_date = max_date - pd.Timedelta(days=7)
        last_complete_year = last_complete_date.isocalendar().year
        last_complete_week = last_complete_date.isocalendar().week
        last_complete_week_str = f"{last_complete_year}-W{str(last_complete_week).zfill(2)}"
        df = df[df['Week'] <= last_complete_week_str]
        print(f"⚠️ Se excluyó la semana incompleta: {max_date.isocalendar().year}-W{str(max_date.isocalendar().week).zfill(2)}")    
 
    # 3. Agrupación semanal
    weekly = (df.groupby(['ItemCode', 'Week', 'StartWeek'])
             .agg({
                 'InQty': 'sum',
                 'OutQty': 'sum'
             })
             .reset_index())
    
    # 4. Cálculo de movimiento neto
    weekly['NetMovement'] = weekly['InQty'] - weekly['OutQty']
    
    # 5. Crear todas las combinaciones de semanas completas
    start_date = weekly['StartWeek'].min()
    end_date = weekly['StartWeek'].max()
    all_weeks = pd.date_range(start=start_date, 
                            end=end_date, 
                            freq='W-MON')
    
    items = weekly['ItemCode'].unique()
    
    # 6. Producto cartesiano de items y semanas
    complete_df = pd.DataFrame(
        [(item, week) for item in items for week in all_weeks],
        columns=['ItemCode', 'StartWeek']
    )
    
    # 7. Añadir formato de semana ISO usando isocalendar
    complete_df['IsoYear'] = complete_df['StartWeek'].dt.isocalendar().year
    complete_df['IsoWeek'] = complete_df['StartWeek'].dt.isocalendar().week
    complete_df['Week'] = complete_df['IsoYear'].astype(str) + '-W' + complete_df['IsoWeek'].astype(str).str.zfill(2)
     
    # 8. Unión con datos existentes
    weekly = pd.merge(
        complete_df,
        weekly[['ItemCode', 'Week', 'InQty', 'OutQty', 'NetMovement']],
        on=['ItemCode', 'Week'],
        how='left'
    )
    
    # 9. Rellenar valores faltantes
    weekly[['InQty', 'OutQty', 'NetMovement']] = weekly[['InQty', 'OutQty', 'NetMovement']].fillna(0)
    
    # 10. Cálculo de stock disponible
    weekly['AvailableStock'] = weekly.groupby('ItemCode')['NetMovement'].cumsum()
    
    # 11. Detección de stockouts
    weekly['Stockout'] = (weekly['AvailableStock'] <= 0).astype(int)
    
    # 12. Ordenar y seleccionar columnas finales
    result = weekly[['ItemCode', 'StartWeek', 'Week', 'InQty', 'OutQty', 
                    'NetMovement', 'AvailableStock', 'Stockout']]
    
    return result.sort_values(['ItemCode', 'StartWeek'])

In [13]:
# Ejemplo de uso
dfWeekStock = process_weekly_inventory(df_oinm)
print("Muestra del inventario procesado (solo semanas completas):")
display(dfWeekStock.head(10))

⚠️ Se excluyó la semana incompleta: 2025-W32
Muestra del inventario procesado (solo semanas completas):


,ItemCode,StartWeek,Week,InQty,OutQty,NetMovement,AvailableStock,Stockout
0,SKU-00750,2017-12-25,2017-W52,102.0,0.0,102.0,102.0,0
1,SKU-00750,2018-01-01,2018-W01,0.0,0.0,0.0,102.0,0
2,SKU-00750,2018-01-08,2018-W02,0.0,0.0,0.0,102.0,0
3,SKU-00750,2018-01-15,2018-W03,0.0,7.0,-7.0,95.0,0
4,SKU-00750,2018-01-22,2018-W04,0.0,0.0,0.0,95.0,0
5,SKU-00750,2018-01-29,2018-W05,0.0,0.0,0.0,95.0,0
6,SKU-00750,2018-02-05,2018-W06,0.0,0.0,0.0,95.0,0
7,SKU-00750,2018-02-12,2018-W07,0.0,0.0,0.0,95.0,0
8,SKU-00750,2018-02-19,2018-W08,0.0,0.0,0.0,95.0,0
9,SKU-00750,2018-02-26,2018-W09,0.0,0.0,0.0,95.0,0


In [14]:
# GUARDAR EL DATAFRAME PROCESADO DFStock
dfWeekStock.to_parquet('../data/silver/dfWeekStock.parquet', index=False)

## 6. CONCLUSIONES

- Se identificaron que los productos mas vendidos son los SKUs 06074, 03543 y 03063.
- El análisis de distribución mostró que las ventas semanales no siguen una distribución normal (p-valor < 0.05 en la prueba de Shapiro-Wilk), lo que es común en datos de ventas del mundo real debido a factores externos como promociones o eventos estacionales.
- El analisis de outliers mostro que hay 2 valoes atipicos en la variable ventas semanales. Es coveniente normalizar estos datos ya que el proximo paso sera crear un modelo de pronostico.